<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Daily_challenge_W5_J4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prédiction du cours des actions avec LSTM et PyTorch

Ce notebook couvre les étapes de préparation des données de séries temporelles, la construction et l'entraînement d'un modèle LSTM avec PyTorch, et l'évaluation de ses performances pour la prédiction du cours des actions.

## 1. Installation des bibliothèques requises

Nous allons installer les bibliothèques nécessaires pour le traitement des données et la construction du modèle PyTorch.

In [ ]:
import sys

%pip install torch pandas numpy scikit-learn matplotlib

## 2. Charger et prétraiter l'ensemble de données

Nous allons charger le fichier `stock_prices.csv`, supprimer les colonnes inutiles, créer une colonne cible et normaliser les données.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

# Téléchargez le fichier 'stock_prices.csv' dans votre environnement Colab
# Si le fichier n'est pas déjà là, vous pouvez le glisser-déposer dans le panneau 'Fichiers' de Colab.

try:
    df = pd.read_csv('stock_prices.csv')
    print('Dataset loaded successfully.')
except FileNotFoundError:
    print("Erreur: 'stock_prices.csv' n'a pas été trouvé. Veuillez télécharger le fichier dans votre environnement Colab.")
    # Créer un DataFrame factice pour que le reste du code puisse fonctionner sans erreur directe si le fichier est manquant.
    # Dans un scénario réel, vous voudriez gérer cette erreur plus robustement.
    data = {
        'Date': pd.to_datetime(pd.date_range(start='2020-01-01', periods=100)),
        'Open': np.random.rand(100) * 100,
        'High': np.random.rand(100) * 100 + 5,
        'Low': np.random.rand(100) * 100 - 5,
        'Close': np.random.rand(100) * 100,
        'Adj Close': np.random.rand(100) * 100,
        'Volume': np.random.randint(100000, 1000000, 100)
    }
    df = pd.DataFrame(data)
    print("Un DataFrame factice a été créé pour continuer l'exécution.")


df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

# Afficher les premières lignes et les informations du DataFrame
display(df.head())
print(df.info())

In [ ]:
# Sélection des caractéristiques pertinentes (prix d'ouverture, de clôture, haut, bas, volume)
features = ['Open', 'High', 'Low', 'Close', 'Volume']
data = df[features].values

# Normalisation des données
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

# Création de la colonne cible (prix de clôture du lendemain)
# Nous décalons la colonne 'Close' d'un jour pour prédire le prix de clôture suivant
close_scaler = MinMaxScaler(feature_range=(0, 1))
close_prices = df['Close'].values.reshape(-1, 1)
scaled_close_prices = close_scaler.fit_transform(close_prices)

# Target (prix de clôture du lendemain)
target = np.roll(scaled_close_prices, -1, axis=0) # Décaler d'un jour en arrière pour que la cible soit le prix du lendemain
target[-1] = target[-2] # Le dernier jour n'a pas de jour suivant, donc nous copions la valeur précédente

print("Données brutes:")
display(df[features].head())
print("Données normalisées (premières 5 lignes):\n", scaled_data[:5])
print("Cible normalisée (premières 5 lignes):\n", target[:5])

## 3. Préparer l'ensemble de données pour l'entraînement

Nous allons diviser l'ensemble de données en ensembles d'entraînement, de validation et de test, créer une classe `Dataset` personnalisée et utiliser `DataLoader`.

In [ ]:
# Fonction pour créer des séquences de données pour le LSTM
def create_sequences(data, target, sequence_length):
    xs, ys = [], []
    for i in range(len(data) - sequence_length):
        x = data[i:(i + sequence_length)]
        y = target[i + sequence_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

SEQUENCE_LENGTH = 30 # Nombre de jours à considérer pour prédire le jour suivant

X, y = create_sequences(scaled_data, target, SEQUENCE_LENGTH)

print(f"Forme de X (entrée): {X.shape}")
print(f"Forme de y (cible): {y.shape}")

# Division des données en ensembles d'entraînement, de validation et de test
train_size = int(len(X) * 0.7)
val_size = int(len(X) * 0.15)
test_size = len(X) - train_size - val_size

X_train, y_train = X[:train_size], y[:train_size]
X_val, y_val = X[train_size:train_size + val_size], y[train_size:train_size + val_size]
X_test, y_test = X[train_size + val_size:], y[train_size + val_size:]

print(f"Taille de l'ensemble d'entraînement: {len(X_train)}")
print(f"Taille de l'ensemble de validation: {len(X_val)}")
print(f"Taille de l'ensemble de test: {len(X_test)}")

In [ ]:
# Définition de la classe Dataset personnalisée
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Création des instances Dataset
train_dataset = StockDataset(X_train, y_train)
val_dataset = StockDataset(X_val, y_val)
test_dataset = StockDataset(X_test, y_test)

# Création des DataLoader
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Nombre de lots d'entraînement: {len(train_loader)}")
print(f"Nombre de lots de validation: {len(val_loader)}")
print(f"Nombre de lots de test: {len(test_loader)}")

## 4. Définir le modèle LSTM

Nous allons créer un modèle LSTM à l'aide de PyTorch, définissant son architecture.

In [ ]:
class StockPredictor(nn.Module):
    def __init__(self, n_features, n_hidden, n_layers, dropout):
        super(StockPredictor, self).__init__()
        self.n_hidden = n_hidden
        self.n_layers = n_layers

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=n_hidden,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout # Ajout du dropout entre les couches LSTM si n_layers > 1
        )

        # Ajout d'une couche dropout après le LSTM et avant la couche linéaire
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(n_hidden, 1)

    def forward(self, x):
        # x shape: (batch_size, sequence_length, n_features)
        # h_0 shape: (n_layers, batch_size, n_hidden)
        # c_0 shape: (n_layers, batch_size, n_hidden)

        h0 = torch.zeros(self.n_layers, x.size(0), self.n_hidden).to(x.device)
        c0 = torch.zeros(self.n_layers, x.size(0), self.n_hidden).to(x.device)

        out, _ = self.lstm(x, (h0, c0))
        # out shape: (batch_size, sequence_length, n_hidden)

        # Prend la sortie de la dernière étape de la séquence
        out = self.dropout(out[:, -1, :])
        out = self.linear(out)
        return out

# Paramètres du modèle
N_FEATURES = len(features)
N_HIDDEN = 128
N_LAYERS = 2
DROPOUT = 0.2

model = StockPredictor(N_FEATURES, N_HIDDEN, N_LAYERS, DROPOUT)

# Définir l'appareil (GPU si disponible, sinon CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(model)

## 5. Entraîner le modèle

Nous allons configurer l'optimiseur, la fonction de perte et exécuter les boucles d'entraînement et de validation.

In [ ]:
EPOCHS = 50
LEARNING_RATE = 0.001

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        model.train()
        running_train_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_train_loss += loss.item() * inputs.size(0)

        epoch_train_loss = running_train_loss / len(train_loader.dataset)
        train_losses.append(epoch_train_loss)

        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                running_val_loss += loss.item() * inputs.size(0)

        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        val_losses.append(epoch_val_loss)

        print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}')

    return train_losses, val_losses

# Lancement de l'entraînement
train_losses, val_losses = train_model(model, train_loader, val_loader, criterion, optimizer, EPOCHS)

# Visualisation de la perte d'entraînement et de validation
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

## 6. Évaluer le modèle

Nous allons calculer le score R² pour évaluer les performances du modèle sur l'ensemble de test et conserver l'objet scaler pour les prédictions futures.

In [ ]:
def evaluate_model(model, test_loader, scaler, device):
    model.eval()
    predictions = []
    actuals = []

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)

            # Inverse la normalisation pour obtenir les valeurs réelles
            predictions.extend(scaler.inverse_transform(outputs.cpu().numpy()).flatten())
            actuals.extend(scaler.inverse_transform(targets.cpu().numpy()).flatten())

    return np.array(predictions), np.array(actuals)

# Obtenir les prédictions et les valeurs réelles
predictions, actuals = evaluate_model(model, test_loader, close_scaler, device)

# Calcul du score R²
r2 = r2_score(actuals, predictions)
print(f"R-squared score on the test set: {r2:.4f}")

# Visualisation des prédictions vs les valeurs réelles
plt.figure(figsize=(14, 7))
plt.plot(actuals, label='Actual Prices', color='blue')
plt.plot(predictions, label='Predicted Prices', color='red', linestyle='--')
plt.title('Stock Price Prediction (Actual vs. Predicted)')
plt.xlabel('Time Step (Test Set)')
plt.ylabel('Stock Price')
plt.legend()
plt.grid(True)
plt.show()

print("Objet scaler (close_scaler) conservé pour les prédictions futures.")